# Index Tracking com Controle de Turnover: S&P 100 e IBOVESPA
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/podorize/index-tracking/blob/main/notebooks/index_tracking_colab_otimizado.ipynb)

Este notebook executa o modelo de **Index Tracking (MIQP)** com **restrição de cardinalidade (máximo K ativos)** e **controle de turnover (giro)** para o **S&P 100** e o **IBOVESPA**.

### Por que este repositório é otimizado?
1. **Zero download repetido:** A base limpa e auditada de 7 anos (1.759 pregões) é carregada do cache do repositório em **0,1 segundo**.
2. **Modo Instantâneo:** Carrega tabelas e gráficos pré-computados dos 20 trimestres fora da amostra imediatamente.
3. **Solver Compacto:** Formulação MIQP que reduz o número de variáveis de 400 para menos de 190, viabilizando resolução em segundos.

In [ ]:
# 1. Configuração do Ambiente no Google Colab
import sys, os
from pathlib import Path

if "google.colab" in sys.modules:
    if not Path("index-tracking").exists():
        !git clone https://github.com/podorize/index-tracking.git
    %cd index-tracking
    %pip install -q gurobipy yfinance

sys.path.append(os.getcwd())
print("Ambiente pronto. Diretório atual:", os.getcwd())


In [ ]:
# 2. Escolha do Modo de Execução
# "instantaneo": visualiza resultados pré-computados em 2 segundos
# "recomputar": re-executa a otimização no Gurobi
MODO = "instantaneo"
print(f"Modo selecionado: {MODO}")


In [ ]:
# 3. Resultados Pré-computados e Fronteira de Pareto
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

df_sp = pd.read_csv("results/tabela_pareto_sp100.csv")
df_ib = pd.read_csv("results/tabela_pareto_ibov.csv")

cols = ["lambda_anual", "TE (% a.a.)", "Giro médio (% trim)", "Custo fricção (% a.a.)", "Retorno anual líquido (%)", "Alfa líquido (p.p. a.a.)"]
print("=== Resultados Fora da Amostra: S&P 100 (c = 5 bps, K = 10) ===")
display(df_sp[cols].round(2))

print("
=== Resultados Fora da Amostra: IBOVESPA (c = 15 bps, K = 10) ===")
display(df_ib[cols].round(2))


In [ ]:
# 4. Visualização da Fronteira de Pareto (TE vs Giro)
fig_path = Path("figures/curva_pareto_turnover.png")
if fig_path.exists():
    img = Image.open(fig_path)
    plt.figure(figsize=(15, 6), dpi=150)
    plt.imshow(img)
    plt.axis("off")
    plt.show()


In [ ]:
# 5. Re-execução Interativa (Apenas se MODO == "recomputar")
if MODO == "recomputar":
    from src.data_loader import preparar_bases
    from src.walk_forward import simular_walk_forward
    
    bases = preparar_bases()
    print("Rodando simulação interativa para IBOVESPA com K=10 e lambda=3.0...")
    metricas, df_janelas = simular_walk_forward("IBOV", bases, K=10, lambda_anual=3.0, custo_transacao=0.0015, tempo_max=1.5)
    print("Métricas obtidas:")
    for k, v in metricas.items():
        print(f"  {k}: {v}")
